# out_of_sample_portfolio_management — all models (OOS, causal)

Same pipeline and rules as the original DeepLOB-only version. The only change
is **which model generates the trading signal**: this now loops the identical
backtest over every model in the project's `_all` checkpoint set (`deeplob`,
`ctabl`, `dla`, `tlob`, `linvar`, `logreg`, `jumpgatelob`, `alphastablelob`) —
the full-universe-trained runs under `checkpoints/stocks/feishu/<model>/*_all/best.pt`.

- Loads **out-of-sample** parquet files (`lob_data_release_stage_out_of_sample.parquet`,
  `daily_data_release_stage_out_of_sample.parquet`) for backtesting, plus the
  **in-sample** files as warmup history for rolling features.
- **All trading days** in the OOS data are backtested (no train/val/test split).
- Feature engineering, causal windowing, `PortfolioManager` (competition
  rules: lot size, commission, stamp duty, T+1 lock, min holdings, buy/sell
  percentage submission format), and `compute_metrics` are **unchanged** from
  the original notebook — only the model producing `(P(down), P(flat), P(up))`
  changes per loop iteration.
- **Causal scoring**: signal for day *t* uses window `[t-T, t-1]` — no peek
  at day-t data when the morning vwap[t] is the entry price.

> Each model runs the full OOS backtest independently (own `PortfolioManager`,
> own trade log, own submission CSV) — this is 8x the runtime of the original
> single-model notebook.

In [1]:
import sys
import random
import logging
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy import stats

import torch
import torch.nn as nn

warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-7s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("portfolio_oos")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "data").is_dir() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
log.info(f"project root: {PROJECT_ROOT}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

for d in ("submissions", "outputs"):
    (PROJECT_ROOT / d).mkdir(exist_ok=True)

01:20:37  INFO     project root: /Users/arshia/Projects/Personal/Penny


In [2]:
CONFIG = {
    # ── paths ─────────────────────────────────────────────────────────────
    # OOS files (what we backtest on)
    "lob_path": str(
        PROJECT_ROOT
        / "data"
        / "stocks"
        / "feishu"
        / "lob_data_release_stage_out_of_sample.parquet"
    ),
    "ohlcv_path": str(
        PROJECT_ROOT
        / "data"
        / "stocks"
        / "feishu"
        / "daily_data_release_stage_out_of_sample.parquet"
    ),
    # In-sample files used only as WARMUP for rolling features (ret_20d, vol_20d,
    # rsi_14, OFI 5-day z-score, …). Causal — same factual history a live
    # trader would have at the start of the OOS period.
    "lob_in_sample_path": str(
        PROJECT_ROOT / "data" / "stocks" / "feishu" / "lob_data_in_sample.parquet"
    ),
    "ohlcv_in_sample_path": str(
        PROJECT_ROOT / "data" / "stocks" / "feishu" / "daily_data_in_sample.parquet"
    ),
    # per-model checkpoints: newest run whose directory name ends in
    # ckpt_variant, under checkpoints/stocks/feishu/<tag>/ — same discovery
    # convention used by notebooks/stocks/feishu/*.ipynb.
    "ckpt_root": str(PROJECT_ROOT / "checkpoints" / "stocks" / "feishu"),
    "ckpt_variant": "all",  # "all" | "20_assets"
    # ── competition rules ─────────────────────────────────────────────────
    "initial_capital": 50_000_000,
    "min_holdings": 10,
    "sell_mode": "close",
    "team_id": "T001",
    "lot_size": 100,
    "commission_rate": 1e-4,
    "stamp_duty_rate": 5e-4,
    "min_commission": 5,
    # ── model / features ──────────────────────────────────────────────────
    "T": 50,
    "NF": 259,
    "N_LEVELS": 10,
    "N_OFI": 240,
    "OFI_NORM_DAYS": 5,
    "ALPHA": 0.015,
    # ── signal thresholds (probability-based selection, no constant top-N) ─
    "min_up_prob": 0.75,  # BUY  every asset with P(up)   > 0.75
    "min_down_prob": 0.85,  # SELL every asset with P(down) > 0.90
    "partial_sell_pct": 0.50,
    # ── cash ──────────────────────────────────────────────────────────────
    "cash_buffer": 0.05,
    # ── metrics ───────────────────────────────────────────────────────────
    "risk_free_annual": 0.0172,
    "trading_days_year": 242,
    # ── execution ─────────────────────────────────────────────────────────
    "batch_size": 64,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

log.info(f"device         : {CONFIG['device']}")
log.info(f"ckpt_root      : {CONFIG['ckpt_root']}  (variant={CONFIG['ckpt_variant']})")
log.info(f"lob (OOS)      : {CONFIG['lob_path']}")
log.info(f"ohlcv (OOS)    : {CONFIG['ohlcv_path']}")
log.info(f"lob (warmup)   : {CONFIG['lob_in_sample_path']}")
log.info(f"ohlcv (warmup) : {CONFIG['ohlcv_in_sample_path']}")

01:20:37  INFO     device         : cpu
01:20:37  INFO     ckpt_root      : /Users/arshia/Projects/Personal/Penny/checkpoints/stocks/feishu  (variant=all)
01:20:37  INFO     lob (OOS)      : /Users/arshia/Projects/Personal/Penny/data/stocks/feishu/lob_data_release_stage_out_of_sample.parquet
01:20:37  INFO     ohlcv (OOS)    : /Users/arshia/Projects/Personal/Penny/data/stocks/feishu/daily_data_release_stage_out_of_sample.parquet
01:20:37  INFO     lob (warmup)   : /Users/arshia/Projects/Personal/Penny/data/stocks/feishu/lob_data_in_sample.parquet
01:20:37  INFO     ohlcv (warmup) : /Users/arshia/Projects/Personal/Penny/data/stocks/feishu/daily_data_in_sample.parquet


In [3]:
def _day_to_int(d: str) -> int:
    return int(d[1:])


def _norm_day_col(df):
    """Standardize date column name to 'trade_day_id'."""
    if "date" in df.columns and "trade_day_id" not in df.columns:
        return df.rename(columns={"date": "trade_day_id"})
    return df


log.info(f"loading OHLCV (warmup): {CONFIG['ohlcv_in_sample_path']}")
ohlcv_warmup = _norm_day_col(pd.read_parquet(CONFIG["ohlcv_in_sample_path"]))

log.info(f"loading OHLCV (OOS)   : {CONFIG['ohlcv_path']}")
ohlcv_oos = _norm_day_col(pd.read_parquet(CONFIG["ohlcv_path"]))

# combined OHLCV: warmup days first, then OOS days; drop any (asset, day) dupes
ohlcv_df = (
    pd.concat([ohlcv_warmup, ohlcv_oos], ignore_index=True)
    .drop_duplicates(subset=["asset_id", "trade_day_id"], keep="last")
    .reset_index(drop=True)
)

# day axes
trading_days = sorted(
    ohlcv_df["trade_day_id"].unique().tolist(), key=_day_to_int
)  # warmup + OOS
oos_days = sorted(
    ohlcv_oos["trade_day_id"].unique().tolist(), key=_day_to_int
)  # backtest range

print(
    f"OHLCV combined : {len(ohlcv_df):,} rows  assets={ohlcv_df['asset_id'].nunique()}"
)
print(
    f"Warmup days    : {trading_days[0]} → {oos_days[0]}  ({len(trading_days) - len(oos_days)} days before OOS)"
)
print(f"OOS days       : {oos_days[0]} → {oos_days[-1]}  (n={len(oos_days)})")
print(
    f"All trading dys: {trading_days[0]} → {trading_days[-1]}  (n={len(trading_days)})"
)

01:20:37  INFO     loading OHLCV (warmup): /Users/arshia/Projects/Personal/Penny/data/stocks/feishu/daily_data_in_sample.parquet
01:20:37  INFO     loading OHLCV (OOS)   : /Users/arshia/Projects/Personal/Penny/data/stocks/feishu/daily_data_release_stage_out_of_sample.parquet


OHLCV combined : 1,606,720 rows  assets=2306
Warmup days    : D001 → D485  (484 days before OOS)
OOS days       : D485 → D726  (n=242)
All trading dys: D001 → D726  (n=726)


In [4]:
import pyarrow.parquet as pq

# ── intraday time grid ────────────────────────────────────────────────────────
INTRADAY_SLOTS = [
    "09:40:00",
    "09:50:00",
    "10:00:00",
    "10:10:00",
    "10:20:00",
    "10:30:00",
    "10:40:00",
    "10:50:00",
    "11:00:00",
    "11:10:00",
    "11:20:00",
    "13:00:00",
    "13:10:00",
    "13:20:00",
    "13:30:00",
    "13:40:00",
    "13:50:00",
    "14:00:00",
    "14:10:00",
    "14:20:00",
    "14:30:00",
    "14:40:00",
    "14:50:00",
    "15:00:00",
]
SLOT_TO_IDX = {t: i for i, t in enumerate(INTRADAY_SLOTS)}
N_SLOTS = len(INTRADAY_SLOTS)  # 24
VWAP_COL = "vwap_0930_0935"


# ── OFI helpers ───────────────────────────────────────────────────────────────
def _bOF(b_price, b_vol):
    up = b_price > b_price.shift(1)
    same = b_price == b_price.shift(1)
    dn = b_price < b_price.shift(1)
    return up * b_vol + same * (b_vol - b_vol.shift(1)) + dn * (-b_vol.shift(1))


def _aOF(a_price, a_vol):
    up = a_price > a_price.shift(1)
    same = a_price == a_price.shift(1)
    dn = a_price < a_price.shift(1)
    return up * (-a_vol.shift(1)) + same * (a_vol - a_vol.shift(1)) + dn * a_vol


def _OFI(bp, bv, ap, av):
    return _bOF(bp, bv) - _aOF(ap, av)


def ofi_to_daily_concat_normed(df, n_levels, norm_days):
    ofi_arr = np.zeros((len(df), n_levels), dtype=np.float32)
    for i in range(n_levels):
        j = i + 1
        ofi = _OFI(
            df[f"bid_price_{j}"],
            df[f"bid_volume_{j}"].fillna(0.0),
            df[f"ask_price_{j}"],
            df[f"ask_volume_{j}"].fillna(0.0),
        ).fillna(0.0)
        ofi_arr[:, i] = ofi.values

    days = df["trade_day_id"].values
    slot = df["time"].map(SLOT_TO_IDX).to_numpy()
    unique_days, inv = np.unique(days, return_inverse=True)
    n_days = len(unique_days)

    grid = np.zeros((n_days, N_SLOTS, n_levels), dtype=np.float32)
    keep = ~pd.isna(slot)
    grid[inv[keep], slot[keep].astype(np.int64), :] = ofi_arr[keep]
    daily = grid.reshape(n_days, N_SLOTS * n_levels)

    n_feat = N_SLOTS * n_levels
    normed = np.empty_like(daily)
    for d in range(n_days):
        past = daily[max(0, d - norm_days) : d]
        if len(past) >= 2:
            mu, std = past.mean(0), past.std(0)
        elif len(past) == 1:
            mu, std = past[0], np.zeros(n_feat, dtype=np.float32)
        else:
            mu = np.zeros(n_feat, dtype=np.float32)
            std = np.ones(n_feat, dtype=np.float32)
        normed[d] = (daily[d] - mu) / (std + 1e-9)

    cols = {f"ofi_{i + 1}": normed[:, i] for i in range(n_feat)}
    cols["date"] = unique_days
    return pd.DataFrame(cols)


# ── OHLCV feature builder ─────────────────────────────────────────────────────
OHLCV_ENG_COLS = [
    "ret_1d",
    "ret_5d",
    "ret_10d",
    "ret_20d",
    "vol_5d",
    "vol_20d",
    "amihud",
    "volume_zscore",
    "rsi_14",
    "ma_dist_5",
    "ma_dist_20",
    "open_close_ret",
    "high_low_range",
    "close_vwap_dist",
]
OHLCV_RAW_COLS = ["raw_open", "raw_close", "raw_volume", "raw_low", "raw_high"]
OHLCV_COLS = OHLCV_ENG_COLS + OHLCV_RAW_COLS


def _build_asset_ohlcv_feats(g):
    c = g["close"].values.astype(np.float64)
    o = g["open"].values.astype(np.float64)
    h = g["high"].values.astype(np.float64)
    lo = g["low"].values.astype(np.float64)
    v = g["volume"].values.astype(np.float64)
    if VWAP_COL in g.columns:
        vw = g[VWAP_COL].values.astype(np.float64)
    elif "vwap" in g.columns:
        vw = g["vwap"].values.astype(np.float64)
    else:
        vw = c.copy()
    n = len(c)

    log_ret = np.empty(n)
    log_ret[0] = 0.0
    log_ret[1:] = np.log(c[1:] / (c[:-1] + 1e-9))

    def ret_n(k):
        r = np.full(n, np.nan)
        r[k:] = c[k:] / (c[:-k] + 1e-9) - 1
        return r

    def roll_std(k):
        return pd.Series(log_ret).rolling(k, min_periods=k).std().values

    def roll_mean(k):
        return pd.Series(c).rolling(k, min_periods=k).mean().values

    delta = pd.Series(c).diff()
    gain = delta.clip(lower=0).rolling(14, min_periods=14).mean()
    loss = (-delta.clip(upper=0)).rolling(14, min_periods=14).mean()
    rsi = (100 - 100 / (1 + gain / (loss + 1e-9))).values

    amihud = np.abs(ret_n(1)) / (v * c + 1e-9)
    vs = pd.Series(v)
    vol_z = (
        (vs - vs.rolling(20, min_periods=5).mean())
        / (vs.rolling(20, min_periods=5).std() + 1e-9)
    ).values

    feats = {
        "ret_1d": ret_n(1),
        "ret_5d": ret_n(5),
        "ret_10d": ret_n(10),
        "ret_20d": ret_n(20),
        "vol_5d": roll_std(5),
        "vol_20d": roll_std(20),
        "amihud": amihud,
        "volume_zscore": vol_z,
        "rsi_14": rsi,
        "ma_dist_5": c / (roll_mean(5) + 1e-9) - 1,
        "ma_dist_20": c / (roll_mean(20) + 1e-9) - 1,
        "open_close_ret": c / (o + 1e-9) - 1,
        "high_low_range": (h - lo) / (c + 1e-9),
        "close_vwap_dist": c / (vw + 1e-9) - 1,
        "raw_open": o,
        "raw_close": c,
        "raw_volume": v,
        "raw_low": lo,
        "raw_high": h,
    }
    out = pd.DataFrame(feats, index=g.index)
    out["asset_id"] = g["asset_id"].values
    out["date"] = g["date"].values
    return out


def _cs_z(s):
    mu, std = s.mean(), s.std()
    return (s - mu) / (std + 1e-9)


def build_ohlcv_features(ohlcv_raw, all_lob_assets, logger):
    ohlcv = ohlcv_raw[ohlcv_raw["asset_id"].isin(set(all_lob_assets))].copy()
    ohlcv = ohlcv.sort_values(["asset_id", "date"]).reset_index(drop=True)
    frames = [
        _build_asset_ohlcv_feats(g)
        for _, g in tqdm(
            ohlcv.groupby("asset_id", sort=False), desc="OHLCV features", unit="asset"
        )
    ]
    feats = pd.concat(frames, ignore_index=True)
    del frames
    for col in OHLCV_COLS:
        feats[col] = feats.groupby("date")[col].transform(_cs_z)
    logger.info(
        f"OHLCV features: {len(feats):,} rows  "
        f"assets={feats['asset_id'].nunique()}  cols={len(OHLCV_COLS)}"
    )
    return feats

In [5]:
# Real model classes from src/models — the exact classes stocks.feishu.train_<tag>
# checkpointed, so `cls(ckpt["config"])` + `load_state_dict(ckpt["model"])` matches
# every "_all" checkpoint's saved state exactly.
from models.alphastablelob import AlphaStableLOB
from models.ctabl import CTABL
from models.deeplob import DeepLOB
from models.dla import DLA
from models.jumpgatelob import JumpGateLOB
from models.linvar import LinVAR
from models.logreg import LogReg
from models.tlob import TLOB

# tag -> model class. checkpoints/stocks/feishu/<tag>/ holds that model's runs.
MODEL_REGISTRY = {
    "logreg": LogReg,
    "linvar": LinVAR,
    "deeplob": DeepLOB,
    "ctabl": CTABL,
    "dla": DLA,
    "tlob": TLOB,
    "jumpgatelob": JumpGateLOB,
    "alphastablelob": AlphaStableLOB,
}
log.info(f"model registry: {list(MODEL_REGISTRY)}")

01:20:37  INFO     model registry: ['logreg', 'linvar', 'deeplob', 'ctabl', 'dla', 'tlob', 'jumpgatelob', 'alphastablelob']


In [6]:
_N = CONFIG["N_LEVELS"]
ASK_P = [f"ask_price_{i}" for i in range(1, _N + 1)]
BID_P = [f"bid_price_{i}" for i in range(1, _N + 1)]
ASK_V = [f"ask_volume_{i}" for i in range(1, _N + 1)]
BID_V = [f"bid_volume_{i}" for i in range(1, _N + 1)]
STREAM_COLS = ["asset_id", "trade_day_id", "time"] + ASK_P + BID_P + ASK_V + BID_V
L1_COLS = ["ask_price_1", "bid_price_1", "ask_volume_1", "bid_volume_1"]
STREAM_BATCH = 500_000

OFI_COLS = [f"ofi_{i + 1}" for i in range(CONFIG["N_OFI"])]
FEAT_COLS = OFI_COLS + OHLCV_COLS
assert len(FEAT_COLS) == CONFIG["NF"]

# ── 1. stream BOTH LOB files (warmup + OOS) into the same per-asset dict ──────
asset_lob_chunks: dict = {}
total_rows = 0

for label, path in (
    ("warmup", CONFIG["lob_in_sample_path"]),
    ("OOS", CONFIG["lob_path"]),
):
    pf = pq.ParquetFile(path)
    print(f"Streaming {label}: {path}  (batch={STREAM_BATCH:,} rows)…")
    file_rows = 0
    for _batch in pf.iter_batches(batch_size=STREAM_BATCH, columns=STREAM_COLS):
        _df = _batch.to_pandas()
        file_rows += len(_df)
        total_rows += len(_df)
        _df = _df[_df[L1_COLS].notna().all(axis=1)]
        for _aid, _grp in _df.groupby("asset_id", sort=False):
            asset_lob_chunks.setdefault(_aid, []).append(_grp)
        print(
            f"  {label} rows={file_rows:,}  total={total_rows:,}  assets={len(asset_lob_chunks)}",
            end="\r",
        )
    print()
del _batch, _df
print(f"Done. total_rows={total_rows:,}  assets={len(asset_lob_chunks)}")

# ── 2. per-asset: concat warmup+OOS chunks → sort by (day, time) → daily OFI ──
# Sorting by trade_day_id puts warmup days before OOS days, which makes the
# 5-day rolling OFI z-score causal across the warmup→OOS boundary.
ofi_frames = []
for asset_id in tqdm(sorted(asset_lob_chunks), desc="Daily OFI", unit="asset"):
    chunks = asset_lob_chunks.pop(asset_id)
    df = (
        pd.concat(chunks, ignore_index=True)
        .sort_values(["trade_day_id", "time"])
        .drop_duplicates(subset=["trade_day_id", "time"], keep="last")
        .reset_index(drop=True)
    )
    del chunks
    if len(df) < 2:
        continue
    daily_df = ofi_to_daily_concat_normed(
        df, CONFIG["N_LEVELS"], CONFIG["OFI_NORM_DAYS"]
    )
    daily_df["asset_id"] = asset_id
    ofi_frames.append(daily_df)
    del df

ofi_daily = pd.concat(ofi_frames, ignore_index=True).rename(
    columns={"date": "trade_day_id"}
)
del ofi_frames, asset_lob_chunks
log.info(f"OFI rows: {len(ofi_daily):,}  assets={ofi_daily['asset_id'].nunique()}")

# ── 3. OHLCV features over the combined timeline ──────────────────────────────
all_lob_assets = sorted(ofi_daily["asset_id"].unique().tolist())
ohlcv_in = ohlcv_df.rename(columns={"trade_day_id": "date"})
ohlcv_feats = build_ohlcv_features(ohlcv_in, all_lob_assets, log).rename(
    columns={"date": "trade_day_id"}
)

# ── 4. merge + attach prices ──────────────────────────────────────────────────
if VWAP_COL in ohlcv_df.columns:
    vwap_col = VWAP_COL
elif "vwap" in ohlcv_df.columns:
    vwap_col = "vwap"
else:
    log.warning("no VWAP column — falling back to open as entry proxy")
    ohlcv_df["_vwap"] = ohlcv_df["open"]
    vwap_col = "_vwap"

daily = (
    ofi_daily.merge(
        ohlcv_feats[["asset_id", "trade_day_id"] + OHLCV_COLS],
        on=["asset_id", "trade_day_id"],
        how="inner",
    )
    .merge(
        ohlcv_df[["asset_id", "trade_day_id", "open", "close", vwap_col]],
        on=["asset_id", "trade_day_id"],
        how="inner",
    )
    .rename(columns={vwap_col: "vwap_entry"})
)
daily["_d_int"] = daily["trade_day_id"].map(_day_to_int)
daily = (
    daily.sort_values(["asset_id", "_d_int"])
    .drop(columns="_d_int")
    .reset_index(drop=True)
)
log.info(f"merged daily: {len(daily):,} rows  assets={daily['asset_id'].nunique()}")

Streaming warmup: /Users/arshia/Projects/Personal/Penny/data/stocks/feishu/lob_data_in_sample.parquet  (batch=500,000 rows)…
  warmup rows=24,810,697  total=24,810,697  assets=2270
Streaming OOS: /Users/arshia/Projects/Personal/Penny/data/stocks/feishu/lob_data_release_stage_out_of_sample.parquet  (batch=500,000 rows)…
  OOS rows=12,761,532  total=37,572,229  assets=2306
Done. total_rows=37,572,229  assets=2306


Daily OFI: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2306/2306 [01:16<00:00, 30.16asset/s]
01:22:04  INFO     OFI rows: 1,601,439  assets=2306
OHLCV features: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2306/2306 [00:02<00:00, 883.56asset/s]
01:22:11  INFO     OHLCV features: 1,606,720 rows  assets=2306  cols=19
01:22:15  INFO     merged daily: 1,601,150 rows  assets=2306


In [7]:
# ── asset → idx: any local enumeration works (model is asset-agnostic) ────────
all_assets = sorted(daily["asset_id"].unique().tolist())
asset_to_idx = {a: i for i, a in enumerate(all_assets)}
n_assets = len(all_assets)
log.info(f"n_assets = {n_assets}")

# ── backtest days: OOS only (warmup days are used as history, not traded) ─────
test_days = oos_days
log.info(f"backtest range: {test_days[0]} → {test_days[-1]}  (n={len(test_days)})")

01:22:15  INFO     n_assets = 2306
01:22:15  INFO     backtest range: D485 → D726  (n=242)


In [8]:
day_to_idx = {d: i for i, d in enumerate(trading_days)}
N_DAYS = len(trading_days)
NF = CONFIG["NF"]

feat = np.zeros((n_assets, N_DAYS, NF), dtype=np.float32)
valid_mask = np.zeros((n_assets, N_DAYS), dtype=bool)
prices = np.full(
    (n_assets, N_DAYS, 3), np.nan, dtype=np.float32
)  # 0=open, 1=close, 2=vwap

a_idx = daily["asset_id"].map(asset_to_idx).to_numpy()
d_idx = daily["trade_day_id"].map(day_to_idx).to_numpy()
feat[a_idx, d_idx, :] = daily[FEAT_COLS].to_numpy(dtype=np.float32)
prices[a_idx, d_idx, 0] = daily["open"].to_numpy(dtype=np.float32)
prices[a_idx, d_idx, 1] = daily["close"].to_numpy(dtype=np.float32)
prices[a_idx, d_idx, 2] = daily["vwap_entry"].to_numpy(dtype=np.float32)
valid_mask[a_idx, d_idx] = True

# Match training preprocessing. Rolling 20-day OHLCV features are NaN for the
# first ~20 days per asset; with warmup history those NaNs land in the warmup
# region but still need to be sanitized before they enter the model.
np.nan_to_num(feat, copy=False, nan=0.0, posinf=5.0, neginf=-5.0)
np.clip(feat, -5.0, 5.0, out=feat)

oos_start = day_to_idx[oos_days[0]]
log.info(f"feature tensor: shape={feat.shape}  size={feat.nbytes / 1e6:.1f} MB")
log.info(
    f"warmup coverage : {valid_mask[:, :oos_start].mean() * 100:.1f}%   ({oos_start} days)"
)
log.info(
    f"OOS coverage    : {valid_mask[:, oos_start:].mean() * 100:.1f}%   ({N_DAYS - oos_start} days)"
)

01:22:17  INFO     feature tensor: shape=(2306, 726, 259)  size=1734.4 MB
01:22:17  INFO     warmup coverage : 94.7%   (484 days)
01:22:17  INFO     OOS coverage    : 97.5%   (242 days)


In [9]:
def discover_checkpoints(ckpt_root: str, variant: str) -> dict:
    """tag -> newest run dir under ckpt_root/<tag>/ whose name ends in `variant`."""
    root = Path(ckpt_root)
    found = {}
    for tag in MODEL_REGISTRY:
        d = root / tag
        if not d.exists():
            continue
        runs = sorted(
            p
            for p in d.iterdir()
            if p.is_dir() and p.name.endswith(variant) and (p / "best.pt").exists()
        )
        if runs:
            found[tag] = runs[-1]
    return found


def load_model(tag: str, device: torch.device) -> nn.Module:
    """Build + load one registered model from its discovered checkpoint."""
    ckpt = torch.load(
        CHECKPOINTS[tag] / "best.pt", map_location=device, weights_only=False
    )
    model = MODEL_REGISTRY[tag](ckpt["config"]).to(device)
    model.load_state_dict(ckpt["model"])
    model.eval()
    return model


CHECKPOINTS = discover_checkpoints(CONFIG["ckpt_root"], CONFIG["ckpt_variant"])
print(f"variant={CONFIG['ckpt_variant']}  |  discovered checkpoints:")
for tag in MODEL_REGISTRY:
    hit = CHECKPOINTS.get(tag)
    print(f"  {tag:<16} {'OK ' + hit.name if hit else '-- missing'}")
if not CHECKPOINTS:
    raise RuntimeError(f"no checkpoints discovered under {CONFIG['ckpt_root']}")

device = torch.device(CONFIG["device"])
print(f"Device         : {device}")
print(f"Input shape    : (B, 1, T={CONFIG['T']}, NF={CONFIG['NF']})")
print("Output classes : 3 [Down, Flat, Up]")

# plumbing smoke test (shape-only, model-agnostic): build the first discovered
# model and confirm the predict()+softmax contract returns valid probabilities.
_probe_tag = next(t for t in MODEL_REGISTRY if t in CHECKPOINTS)
_probe = load_model(_probe_tag, device)
with torch.no_grad():
    _dummy = {"x": torch.zeros(2, 1, CONFIG["T"], CONFIG["NF"], device=device)}
    _out = torch.softmax(_probe.predict(_dummy, device), dim=1)
print(
    f"Smoke test ({_probe_tag}): output={tuple(_out.shape)}  row-sums={_out.sum(1).tolist()}"
)
del _probe
if device.type == "cuda":
    torch.cuda.empty_cache()

variant=all  |  discovered checkpoints:
  logreg           OK logreg_ofi_all
  linvar           OK linvar_ofi_all
  deeplob          OK deeplob_ofi_all
  ctabl            OK ctabl_ofi_all
  dla              OK dla_ofi_all
  tlob             OK tlob_ofi_all
  jumpgatelob      OK jumpgatelob_levy_ofi_all
  alphastablelob   OK alphastablelob_joint_a1.5_ofi_all
Device         : cpu
Input shape    : (B, 1, T=50, NF=259)
Output classes : 3 [Down, Flat, Up]
Smoke test (logreg): output=(2, 3)  row-sums=[1.0, 1.0]


In [10]:
def generate_signals(day_id: str, model: nn.Module) -> pd.DataFrame:
    """
    For decision day t (= `day_id`), use the feature window [t-T, t-1] — i.e.
    everything fully observable by end-of-day-(t-1) — to predict the trade
    made at day-t's morning vwap (entry at vwap[t], exit at close[t+1]).

    Window/pairing logic is unchanged from the single-model notebook: window
    ending at day τ is paired with the trade on day τ+1. Setting τ = t-1 keeps
    the trader at 09:35 of day t with no peek at day-t data. The only change
    is that the model producing the probabilities is now a parameter (called
    through its own `.predict(batch, device)` contract) instead of a
    hardcoded global forward pass.
    """
    t = day_to_idx[day_id]
    if t < CONFIG["T"]:
        return pd.DataFrame(
            columns=[
                "asset_id",
                "trade_day_id",
                "prob_down",
                "prob_flat",
                "prob_up",
                "signal",
            ]
        )

    start = t - CONFIG["T"]  # window: [t-T, t-1] inclusive
    valid = valid_mask[:, start:t].all(axis=1)  # all T days must be present
    idxs = np.where(valid)[0]
    if idxs.size == 0:
        return pd.DataFrame(
            columns=[
                "asset_id",
                "trade_day_id",
                "prob_down",
                "prob_flat",
                "prob_up",
                "signal",
            ]
        )

    X = feat[idxs, start:t, :]  # (n, T, NF) — exclusive of day t
    Xt = torch.from_numpy(X[:, None, :, :]).to(device)  # (n, 1, T, NF)

    chunks = []
    with torch.no_grad():
        for s in range(0, len(idxs), CONFIG["batch_size"]):
            e = s + CONFIG["batch_size"]
            logits = model.predict({"x": Xt[s:e]}, device)
            chunks.append(torch.softmax(logits, dim=1).cpu().numpy())
    probs = np.concatenate(chunks, axis=0)

    sig = pd.DataFrame(
        {
            "asset_id": [all_assets[i] for i in idxs],
            "trade_day_id": day_id,
            "prob_down": probs[:, 0],
            "prob_flat": probs[:, 1],
            "prob_up": probs[:, 2],
        }
    )
    sig["signal"] = probs.argmax(1) - 1
    return sig.sort_values("prob_up", ascending=False).reset_index(drop=True)

In [11]:
class PortfolioManager:
    def __init__(self, config: dict):
        self.cfg = config
        self.cash = float(config["initial_capital"])
        self.holdings = {}
        self.t1_locked = {}
        self.daily_log = []
        self.submission_rows = []

    def sellable_shares(self, aid: str) -> int:
        return self.holdings.get(aid, 0) - self.t1_locked.get(aid, 0)

    def portfolio_value(self, close_prices: dict) -> float:
        return self.cash + sum(
            n * close_prices.get(aid, 0.0) for aid, n in self.holdings.items()
        )

    def num_holdings(self) -> int:
        return sum(1 for n in self.holdings.values() if n > 0)

    def _buy_cost(self, turnover):
        return max(turnover * self.cfg["commission_rate"], self.cfg["min_commission"])

    def _sell_cost(self, turnover):
        commission = max(
            turnover * self.cfg["commission_rate"], self.cfg["min_commission"]
        )
        return commission + turnover * self.cfg["stamp_duty_rate"]

    def _execute_sell(self, aid, sell_pct, price):
        if price <= 0 or np.isnan(price):
            return None
        sellable = self.sellable_shares(aid)
        if sellable <= 0:
            return None
        lot = self.cfg["lot_size"]
        quantity = (
            sellable if sell_pct >= 1.0 else (int(sellable * sell_pct // lot) * lot)
        )
        if quantity <= 0:
            return None
        turnover = quantity * price
        cost = self._sell_cost(turnover)
        self.cash += turnover - cost
        self.holdings[aid] -= quantity
        if self.holdings[aid] == 0:
            del self.holdings[aid]
        return {
            "asset_id": aid,
            "shares_sold": quantity,
            "sell_price": price,
            "turnover": turnover,
            "cost": cost,
            "sell_pct": sell_pct,
        }

    def _execute_buy(self, aid, buy_pct, price):
        if price <= 0 or np.isnan(price):
            return None
        allocated = self.cash * buy_pct
        lot = self.cfg["lot_size"]
        quantity = (int(allocated // price) // lot) * lot
        if quantity < lot:
            return None
        while True:
            turnover = quantity * price
            cost = self._buy_cost(turnover)
            if turnover + cost <= self.cash or quantity < lot:
                break
            quantity -= lot
        if quantity < lot:
            return None
        turnover = quantity * price
        cost = self._buy_cost(turnover)
        self.cash -= turnover + cost
        self.holdings[aid] = self.holdings.get(aid, 0) + quantity
        self.t1_locked[aid] = self.t1_locked.get(aid, 0) + quantity
        return {
            "asset_id": aid,
            "shares_bought": quantity,
            "buy_price": price,
            "turnover": turnover,
            "cost": cost,
            "buy_pct": buy_pct,
        }

    def step(self, day_id: str, signals: pd.DataFrame, ohlcv_day: dict):
        self.t1_locked = {}

        sell_key = "open" if self.cfg["sell_mode"] == "open" else "close"
        sell_prices = {a: row[sell_key] for a, row in ohlcv_day.items()}
        buy_prices = {a: row["vwap_entry"] for a, row in ohlcv_day.items()}
        close_prices = {a: row["close"] for a, row in ohlcv_day.items()}

        sells_done, buys_done = [], []
        sold_today_ids = set()

        # ── SELL (every held asset with P(down) > min_down_prob) ─────────────
        held = [a for a in self.holdings if self.sellable_shares(a) > 0]
        sig_h = signals[signals["asset_id"].isin(held)]
        full_sell_candidates = sig_h[
            (sig_h["prob_down"] > self.cfg["min_down_prob"]) & (sig_h["signal"] == -1)
        ].sort_values("prob_down", ascending=False)
        max_sellable = max(0, self.num_holdings() - self.cfg["min_holdings"])
        full_sell_candidates = full_sell_candidates.head(max_sellable)

        for _, row in full_sell_candidates.iterrows():
            aid = row["asset_id"]
            res = self._execute_sell(aid, 1.0, sell_prices.get(aid, np.nan))
            if res is not None:
                sells_done.append(res)
                sold_today_ids.add(aid)
                self.submission_rows.append(
                    {
                        "trade_day_id": day_id,
                        "asset_id": aid,
                        "buy_percentage": 0.0,
                        "sell_percentage": 1.0,
                    }
                )

        flat_candidates = sig_h[
            (sig_h["signal"] == 0) & (~sig_h["asset_id"].isin(sold_today_ids))
        ]
        for _, row in flat_candidates.iterrows():
            aid = row["asset_id"]
            res = self._execute_sell(
                aid, self.cfg["partial_sell_pct"], sell_prices.get(aid, np.nan)
            )
            if res is None:
                continue
            sells_done.append(res)
            sold_today_ids.add(aid)
            self.submission_rows.append(
                {
                    "trade_day_id": day_id,
                    "asset_id": aid,
                    "buy_percentage": 0.0,
                    "sell_percentage": float(self.cfg["partial_sell_pct"]),
                }
            )

        # ── BUY (every asset with P(up) > min_up_prob, equal weights) ────────
        buy_cands = signals[
            (signals["prob_up"] > self.cfg["min_up_prob"])
            & (signals["signal"] == 1)
            & (~signals["asset_id"].isin(sold_today_ids))
        ].copy()
        buy_cands = buy_cands[
            buy_cands["asset_id"].map(lambda a: not np.isnan(buy_prices.get(a, np.nan)))
        ]

        if len(buy_cands) > 0:
            usable_pct = 1.0 - self.cfg["cash_buffer"]
            equal_w = usable_pct / len(buy_cands)
            buy_cands = buy_cands.assign(_w=equal_w).sort_values(
                "prob_up", ascending=False
            )
            for _, row in buy_cands.iterrows():
                aid = row["asset_id"]
                res = self._execute_buy(
                    aid, float(row["_w"]), buy_prices.get(aid, np.nan)
                )
                if res is None:
                    continue
                buys_done.append(res)
                self.submission_rows.append(
                    {
                        "trade_day_id": day_id,
                        "asset_id": aid,
                        "buy_percentage": float(row["_w"]),
                        "sell_percentage": 0.0,
                    }
                )

        port_val = self.portfolio_value(close_prices)
        self.daily_log.append(
            {
                "trade_day_id": day_id,
                "portfolio_value": port_val,
                "cash": self.cash,
                "num_holdings": self.num_holdings(),
                "num_buys": len(buys_done),
                "num_sells": len(sells_done),
                "buy_turnover": sum(b["turnover"] for b in buys_done),
                "sell_turnover": sum(s["turnover"] for s in sells_done),
                "total_costs": sum(b["cost"] for b in buys_done)
                + sum(s["cost"] for s in sells_done),
            }
        )

    def get_daily_df(self) -> pd.DataFrame:
        return pd.DataFrame(self.daily_log).set_index("trade_day_id")

    def get_submission_df(self) -> pd.DataFrame:
        df = pd.DataFrame(self.submission_rows)
        if df.empty:
            return df
        df = df[~((df["buy_percentage"] == 0) & (df["sell_percentage"] == 0))]
        df = df.drop_duplicates(subset=["trade_day_id", "asset_id"], keep="last")
        df["buy_percentage"] = df["buy_percentage"].round(6)
        df["sell_percentage"] = df["sell_percentage"].round(6)
        return df.sort_values(["trade_day_id", "asset_id"]).reset_index(drop=True)

In [ ]:
runnable = [d for d in test_days if day_to_idx[d] >= CONFIG["T"]]
log.info(
    f"backtesting {len(runnable)} OOS days per model: {runnable[0]} → {runnable[-1]}"
)


def _ohlcv_dict_for_day(day_id: str) -> dict:
    t = day_to_idx[day_id]
    out = {}
    for a_i in np.where(valid_mask[:, t])[0]:
        out[all_assets[a_i]] = {
            "open": float(prices[a_i, t, 0]),
            "close": float(prices[a_i, t, 1]),
            "vwap_entry": float(prices[a_i, t, 2]),
        }
    return out


PM = {}  # tag -> PortfolioManager (post-backtest)
DAILY_DF = {}  # tag -> daily_df

for tag in MODEL_REGISTRY:
    if tag not in CHECKPOINTS:
        print(f"  {tag:<16} -- no checkpoint, skipped")
        continue
    print(f"\n=== {tag} ===")
    model = load_model(tag, device)

    # identical PortfolioManager + backtest loop as the single-model notebook,
    # just re-instantiated per model and fed that model's own signals
    pm = PortfolioManager(CONFIG)
    for day_id in tqdm(runnable, desc=f"backtest[{tag}]", unit="day"):
        sig = generate_signals(day_id, model)
        ohlcv_day = _ohlcv_dict_for_day(day_id)
        pm.step(day_id, sig, ohlcv_day)

    daily_df = pm.get_daily_df()
    final_val = daily_df["portfolio_value"].iloc[-1]
    PM[tag], DAILY_DF[tag] = pm, daily_df

    print(f"OOS days run  : {len(daily_df)}")
    print(f"Final value   : RMB {final_val / 1e6:.2f}M")
    print(f"Total return  : {(final_val / CONFIG['initial_capital'] - 1) * 100:.2f}%")
    min_h = int(daily_df["num_holdings"].min())
    under = int((daily_df["num_holdings"] < CONFIG["min_holdings"]).sum())
    print(
        f"Min holdings  : {min_h}"
        + ("  WARNING" if min_h < CONFIG["min_holdings"] else "")
    )
    print(f"Days < {CONFIG['min_holdings']}    : {under}  (should be 0)")

    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()

print(f"\nbacktested {len(PM)} models")

01:22:17  INFO     backtesting 242 OOS days per model: D485 → D726



=== logreg ===


backtest[logreg]: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 242/242 [00:04<00:00, 54.38day/s]


OOS days run  : 242
Final value   : RMB 64.10M
Total return  : 28.20%
Min holdings  : 3  WARNING
Days < 10    : 7  (should be 0)

=== linvar ===


backtest[linvar]: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 242/242 [00:08<00:00, 28.36day/s]


OOS days run  : 242
Final value   : RMB 30.67M
Total return  : -38.66%
Min holdings  : 0  WARNING
Days < 10    : 83  (should be 0)

=== deeplob ===


backtest[deeplob]:  15%|████████████████████████████▍                                                                                                                                                             | 37/242 [10:13<58:01, 16.98s/day]

In [ ]:
def compute_metrics(daily_values, daily_df, cfg):
    daily_returns = daily_values.pct_change().dropna()
    T_years = max(len(daily_values) / cfg["trading_days_year"], 1e-9)
    rf_daily = (1 + cfg["risk_free_annual"]) ** (1 / cfg["trading_days_year"]) - 1

    cagr = (daily_values.iloc[-1] / daily_values.iloc[0]) ** (1 / T_years) - 1
    excess = daily_returns - rf_daily
    sharpe = (excess.mean() / (excess.std() + 1e-12)) * np.sqrt(
        cfg["trading_days_year"]
    )
    roll_peak = daily_values.cummax()
    drawdowns = (roll_peak - daily_values) / roll_peak
    mdd = float(drawdowns.max())

    return {
        "cagr": float(cagr),
        "sharpe": float(sharpe),
        "mdd": mdd,
        "total_return": float(daily_values.iloc[-1] / daily_values.iloc[0] - 1),
        "ann_vol": float(daily_returns.std() * np.sqrt(cfg["trading_days_year"])),
        "calmar": float(cagr / mdd) if mdd > 0 else float("nan"),
        "total_costs": float(daily_df["total_costs"].sum()),
        "avg_turnover": float(
            (daily_df["buy_turnover"] + daily_df["sell_turnover"]).mean()
        ),
        "win_rate": float((daily_returns > 0).mean()),
        "avg_holdings": float(daily_df["num_holdings"].mean()),
        "drawdowns": drawdowns,
        "daily_returns": daily_returns,
    }


METRICS = {}
for tag, daily_df in DAILY_DF.items():
    m = compute_metrics(daily_df["portfolio_value"], daily_df, CONFIG)
    m["score_proxy"] = (
        0.45 * m["cagr"] * 100 + 0.30 * m["sharpe"] * 10 + 0.25 * (-m["mdd"] * 100)
    )
    METRICS[tag] = m

    print(f"\n----- {tag} -----")
    print("╬" + "═" * 58 + "╬")
    print("║        COMPETITION METRICS  [OUT OF SAMPLE]           ║")
    print("╠" + "═" * 58 + "╣")
    print(f"║  CAGR                        :  {m['cagr'] * 100:>8.2f}%               ║")
    print(f"║  Annualized Sharpe Ratio     :  {m['sharpe']:>8.2f}                ║")
    print(f"║  Maximum Drawdown            :  {-m['mdd'] * 100:>8.2f}%               ║")
    print("╠" + "═" * 58 + "╣")
    print(
        f"║  Score proxy (raw, unranked) :  {m['score_proxy']:>8.2f}                ║"
    )
    print("╠" + "═" * 58 + "╣")
    print(
        f"║  Total Return                :  {m['total_return'] * 100:>8.2f}%               ║"
    )
    print(
        f"║  Annualized Volatility       :  {m['ann_vol'] * 100:>8.2f}%               ║"
    )
    print(f"║  Calmar Ratio                :  {m['calmar']:>8.2f}                ║")
    print(f"║  Total Transaction Costs     :  RMB {m['total_costs']:>12,.0f}    ║")
    print(f"║  Avg Daily Turnover          :  RMB {m['avg_turnover']:>12,.0f}    ║")
    print(
        f"║  Win Rate                    :  {m['win_rate'] * 100:>8.2f}%               ║"
    )
    print(
        f"║  Avg Holdings per Day        :  {m['avg_holdings']:>8.1f}                ║"
    )
    print("╝" + "═" * 58 + "╚")

# cross-model comparison
summary = pd.DataFrame(
    {
        tag: {k: v for k, v in m.items() if k not in ("drawdowns", "daily_returns")}
        for tag, m in METRICS.items()
    }
).T.sort_values("sharpe", ascending=False)
print("\n=== model comparison (sorted by Sharpe) ===")
summary

In [ ]:
def plot_report(tag, daily_df, metrics, cfg):
    fig, axes = plt.subplots(3, 2, figsize=(15, 12))

    ax = axes[0, 0]
    ax.plot(daily_df.index, daily_df["portfolio_value"] / 1e6, color="navy", lw=1.5)
    ax.axhline(cfg["initial_capital"] / 1e6, color="gray", ls="--", alpha=0.6)
    ax.set_title("Portfolio Value (OOS)")
    ax.set_ylabel("RMB (M)")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(alpha=0.3)

    ax = axes[0, 1]
    r = metrics["daily_returns"]
    ax.hist(r, bins=50, density=True, alpha=0.7, color="steelblue", edgecolor="white")
    if len(r) > 1:
        kx = np.linspace(r.min(), r.max(), 200)
        ax.plot(kx, stats.gaussian_kde(r)(kx), color="darkred", lw=1.5)
    ax.axvline(r.mean(), color="black", ls="--", alpha=0.7)
    ax.set_title("Daily Returns Distribution")
    ax.set_xlabel("daily return")
    ax.text(
        0.02,
        0.95,
        f"mean={r.mean() * 100:.2f}%  std={r.std() * 100:.2f}%  skew={r.skew():.2f}",
        transform=ax.transAxes,
        va="top",
        fontsize=9,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
    )
    ax.grid(alpha=0.3)

    ax = axes[1, 0]
    dd = metrics["drawdowns"] * -100
    ax.fill_between(dd.index, dd, 0, color="red", alpha=0.4)
    ax.plot(dd.index, dd, color="darkred", lw=0.8)
    mdd_day = dd.idxmin()
    ax.annotate(
        f"max DD: {dd.min():.1f}%",
        xy=(mdd_day, dd.min()),
        xytext=(0.3, 0.2),
        textcoords="axes fraction",
        arrowprops=dict(arrowstyle="->"),
    )
    ax.set_title("Drawdown Curve")
    ax.set_ylabel("drawdown (%)")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(alpha=0.3)

    ax = axes[1, 1]
    r_idx = pd.Series(metrics["daily_returns"].values, index=daily_df.index[1:])
    roll = r_idx.rolling(20)
    rs = (roll.mean() / (roll.std() + 1e-12)) * np.sqrt(cfg["trading_days_year"])
    ax.plot(rs.index, rs, color="seagreen", lw=1.2)
    ax.axhline(0, color="black", ls="--", alpha=0.6)
    ax.set_title("Rolling 20-Day Sharpe")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(alpha=0.3)

    ax = axes[2, 0]
    colors = [
        "red" if h < cfg["min_holdings"] else "steelblue"
        for h in daily_df["num_holdings"]
    ]
    ax.bar(range(len(daily_df)), daily_df["num_holdings"], color=colors, width=1.0)
    ax.axhline(
        cfg["min_holdings"],
        color="red",
        ls="--",
        lw=1.5,
        label=f"min={cfg['min_holdings']}",
    )
    ax.set_title("Number of Holdings per Day")
    ax.set_xlabel("OOS day idx")
    ax.set_ylabel("# holdings")
    ax.legend()
    ax.grid(alpha=0.3)

    ax = axes[2, 1]
    ax.bar(
        range(len(daily_df)),
        daily_df["total_costs"],
        color="purple",
        width=1.0,
        alpha=0.6,
        label="daily",
    )
    ax.set_xlabel("OOS day idx")
    ax.set_ylabel("daily cost (RMB)")
    ax2 = ax.twinx()
    ax2.plot(
        range(len(daily_df)),
        daily_df["total_costs"].cumsum(),
        color="darkred",
        lw=1.5,
        label="cumulative",
    )
    ax2.set_ylabel("cumulative cost (RMB)")
    ax.set_title("Daily Transaction Costs")
    ax.grid(alpha=0.3)

    fig.suptitle(f"OOS backtest report — {tag}", y=1.01)
    plt.tight_layout()
    out_png = PROJECT_ROOT / "outputs" / f"backtest_report_oos_{tag}.png"
    plt.savefig(out_png, dpi=150, bbox_inches="tight")
    print(f"saved → {out_png}")
    plt.show()


for tag, daily_df in DAILY_DF.items():
    plot_report(tag, daily_df, METRICS[tag], CONFIG)

# cross-model comparison: cumulative portfolio value, all models overlaid
fig, ax = plt.subplots(figsize=(11, 5))
for tag, daily_df in DAILY_DF.items():
    ax.plot(range(len(daily_df)), daily_df["portfolio_value"] / 1e6, lw=1.3, label=tag)
ax.axhline(CONFIG["initial_capital"] / 1e6, color="gray", ls="--", alpha=0.6)
ax.set_xlabel("OOS day idx")
ax.set_ylabel("portfolio value (RMB M)")
ax.set_title("OOS portfolio value — all models")
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout()
out_png = PROJECT_ROOT / "outputs" / "backtest_report_oos_comparison.png"
plt.savefig(out_png, dpi=150, bbox_inches="tight")
print(f"saved → {out_png}")
plt.show()

In [ ]:
for tag, pm in PM.items():
    sub_df = pm.get_submission_df()
    sub_df = sub_df[["trade_day_id", "asset_id", "buy_percentage", "sell_percentage"]]

    assert sub_df[["trade_day_id", "asset_id"]].duplicated().sum() == 0
    assert sub_df["buy_percentage"].between(0, 1).all()
    assert sub_df["sell_percentage"].between(0, 1).all()
    assert not (
        (sub_df["buy_percentage"] == 0) & (sub_df["sell_percentage"] == 0)
    ).any()
    assert not ((sub_df["buy_percentage"] > 0) & (sub_df["sell_percentage"] > 0)).any()
    daily_df = DAILY_DF[tag]
    for day, _ in sub_df.groupby("trade_day_id"):
        if (
            day in daily_df.index
            and daily_df.loc[day, "num_holdings"] < CONFIG["min_holdings"]
        ):
            raise AssertionError(
                f"{tag}  day {day}: holdings < {CONFIG['min_holdings']}"
            )
    print(f"[{tag}] ✅ validation passed")

    fname = (
        PROJECT_ROOT
        / "submissions"
        / f"{CONFIG['team_id']}_oos_sell_{CONFIG['sell_mode']}_{tag}.csv"
    )
    sub_df.to_csv(fname, index=False)
    print(f"[{tag}] trade log saved: {fname}")
    print(
        f"[{tag}] rows: {len(sub_df)}  days: {sub_df['trade_day_id'].nunique()}  "
        f"assets: {sub_df['asset_id'].nunique()}"
    )